In [ ]:
from bs4 import BeautifulSoup
from matplotlib.colors import ListedColormap

from geofeatureviz.io import helpers

There is an [official template provided by Wikipedia](https://upload.wikimedia.org/wikipedia/commons/b/b2/Maps_template-en.svg) for how maps should look. I downloaded this template, which is an SVG-file. Since SVG-files are in general XML-files, I could use BeautifulSoup to read it in. In Inkscape, I checked the group in which the rectangles are located, which has ID "g23682-6". Here, I take the fill colors of all rectangles in this group, sorted by its y-value.

In [ ]:
wiki_template_path = (
    helpers.get_top_directory() / "data" / "Wikipedia_Maps_template-en.svg"
)

with wiki_template_path.open(encoding="utf-8") as f:
    soup = BeautifulSoup(f, "xml")

group = soup.find("g", {"id": "g23682-6"})
assert group is not None

fill_colors = []
for el in group.find_all():
    style = str(el.get("style"))
    style_dict = dict([s.split(":") for s in style.split(";")])
    fill = style_dict["fill"]
    y = el.get("y")

    fill_colors.append((y, fill))
# sort by y-value
fill_colors = [c[1] for c in sorted(fill_colors, key=lambda t: t[0])]
ListedColormap(fill_colors)

For usage, it is necessary to understand the colors and for what they should be used. There is a huge difference between colors used for land and for water. Even on land, there can be depressions below sea level; therefore, there is a color that stands for depression. I distinguish between land and water colors and give the levels names using integers. I print the results to be able to copy them and save them to map_style.yaml as colors.

In [ ]:
land_depression_idx = fill_colors.index(
    "#a7dfd2"
)  # neutral Wikipedia topo color from svg

land_colors = fill_colors[: land_depression_idx + 1]
topo_str_land = [
    f'{f"topo:land:{len(land_colors) - i - 2}: ":16}"{color}"'
    for i, color in enumerate(land_colors)
]

water_colors = fill_colors[land_depression_idx + 1 :]
topo_str_water = [
    f'{f"topo:water:{-i}: ":16}"{color}"' for i, color in enumerate(water_colors)
]

topo_str = topo_str_land + topo_str_water
for s in topo_str:
    print(s)

In [ ]:
ListedColormap(land_colors)

In [ ]:
ListedColormap(water_colors)